In [18]:
from typing import Dict, List

In [19]:
INCH_TO_CM = 2.54
ARM_LENGTH_RATIO = 0.44

In [34]:
class DataSanitizer:

    def __init__(
        self,
        measurements: Dict[str, float]
    ):

        self.measurements = measurements
        self.normalized = {}
        self.outliers = []

    def normalize_units(self) -> Dict[str, float]:

        for key, value in self.measurements.items():

            if value <= 0:

                raise ValueError(
                    f"Invalid value for {key}"
                )

            value_cm = value

            if key == "Height":

                if value < 100:
                    value_cm = value * INCH_TO_CM

            else:

                if value < 50:
                    value_cm = value * INCH_TO_CM

            self.normalized[key] = round(
                value_cm,
                2
            )

        return self.normalized

    def validate_proportions(self) -> List[str]:

        height = self.normalized.get("Height")
        chest = self.normalized.get("Chest")
        waist = self.normalized.get("Waist")
        hip = self.normalized.get("Hip")

        if waist and height and waist > height:

            self.outliers.append(
                "Waist measurement exceeds height."
            )

        if chest and height and chest < 0.30 * height:

            self.outliers.append(
                "Chest measurement unusually small."
            )

        if hip and height and hip > 0.90 * height:

            self.outliers.append(
                "Hip measurement unusually large."
            )

        if waist and height and waist < 0.20 * height:

            self.outliers.append(
                "Waist measurement unusually small."
            )

        return self.outliers

    def estimate_missing(self) -> Dict[str, float]:

        estimated_output = self.normalized.copy()

        height = self.normalized.get("Height")

        if (
            "Arm_Length" not in estimated_output
            and height
        ):

            estimated_arm = (
                ARM_LENGTH_RATIO * height
            )

            estimated_output["Arm_Length"] = round(
                estimated_arm,
                2
            )

        return estimated_output

    def process(self):

        normalized_data = self.normalize_units()

        outliers = self.validate_proportions()

        final_output = self.estimate_missing()

        return {
            "Normalized_Data": normalized_data,
            "Outliers": outliers,
            "Final_Output": final_output
        }

In [35]:
if __name__ == "__main__":

    try:

        measurements = {

            "Height": 70,
            "Chest": 38,
            "Waist": 34,
            "Hip": 40

        }

        sanitizer = DataSanitizer(
            measurements
        )

        results = sanitizer.process()

        print("\nNormalized Data:")
        print(results["Normalized_Data"])

        print("\nOutliers:")

        if results["Outliers"]:

            for issue in results["Outliers"]:
                print("-", issue)

        else:

            print(
                "No abnormal body proportions detected."
            )

        print("\nFinal Output:")
        print(results["Final_Output"])

    except Exception as error:

        print(
            f"Error occurred: {error}"
        )


Normalized Data:
{'Height': 177.8, 'Chest': 96.52, 'Waist': 86.36, 'Hip': 101.6}

Outliers:
No abnormal body proportions detected.

Final Output:
{'Height': 177.8, 'Chest': 96.52, 'Waist': 86.36, 'Hip': 101.6, 'Arm_Length': 78.23}
